# s08 — Ask the user

**What this teaches:** how the agent can *pause* the loop and ask a structured question (text / single / multi / confirm) instead of guessing. Wired through the `askuser` registry and an installed asker (stdin here), the `ask_user` tool turns the human into a participant inside the runner's loop.

**Concept anchor:** the loop from [s01_loop](../s01_loop/s01_loop.ipynb) is unchanged — `ask_user` is just another tool. What's new is who *answers* it: not the model, not the OS, but you.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [s01_loop](../s01_loop/s01_loop.ipynb)).
- **Stdin must be live.** The stdin asker reads a line from `os.Stdin`; if you Run-All the notebook in one shot Jupyter will not forward your keystrokes. Execute the cells one-by-one so the kernel can prompt you for an answer when the agent calls `ask_user`.

## 1. Imports

On top of the usual `agentkit` + `stream`, we pull in `fstools` for the tool catalogue, `askuser` for the registry/asker wiring, and the ADK `tool` package so we can filter the toolset down to just `ask_user`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
    "github.com/blouargant/yoke/internal/askuser"
    "google.golang.org/adk/tool"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Wire the askuser registry

`askuser.NewRegistry()` is the routing table: every `ask_user` tool call lands here and is dispatched to whichever *asker* is installed. `InstallStdinAsker(reg)` plugs in the console reader — the question is rendered on stdout and the user's reply is read from stdin.

Swap this for an HTTP asker, a Web UI bridge, or a Slack adapter and the exact same agent suddenly asks humans on a different channel — the loop doesn't care.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

reg := askuser.NewRegistry()
askuser.InstallStdinAsker(reg)
fmt.Println("registry ready; stdin asker installed")

## 4. Narrow the toolset to just `ask_user`

`fstools.New()` returns the standard read/write/grep/glob/bash toolbox. `fstools.NewAskUserTool(reg)` wraps the registry as a tool. We append it then strip everything else, so the model has only one option: ask. This makes the demo deterministic — without the filter the model is likely to greet itself.

In [ ]:
withAskUserOnly := func(in []tool.Tool) []tool.Tool {
    out := make([]tool.Tool, 0, 1)
    for _, t := range in {
        if t.Name() == "ask_user" {
            out = append(out, t)
        }
    }
    return out
}

tools := append(fstools.New(), fstools.NewAskUserTool(reg))
tools = withAskUserOnly(tools)
fmt.Println("tools available:", len(tools))

## 5. Build the agent + runner

The instruction pins the behaviour: ask exactly once with `kind="text"`, then greet using the answer. The runner is otherwise identical to [s01_loop](../s01_loop/s01_loop.ipynb).

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s08_ask_user",
    Description: "Interactive ask_user tool demo.",
    Model:       llm,
    Instruction: "Use the ask_user tool exactly once to learn the user's name " +
        "(kind=\"text\"), then greet them by that name in a single sentence. " +
        "Do not call any other tool.",
    Tools: tools,
})
must(err)
r, err := agentkit.Runner("s08", a)
must(err)
fmt.Println("agent + runner ready")

## 6. Run — the loop blocks on you

When you execute this cell the model will issue a single `ask_user` tool call. The runner suspends the loop, the stdin asker prints the question, and the kernel waits for your keystroke. After you press Enter the runner resumes and the model produces the greeting.

In [ ]:
prompt := "Please greet me — but find out my name first."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The stream pauses mid-turn: model → `ask_user` tool call → human input → model. This is the same shape as a bash call in [s05_tools](../s05_tools/s05_tools.ipynb), only the "executor" is you.
- Try answering with a long sentence — the model uses the *whole* string, not just a first token, because `kind="text"` is a free-form field.

## Try it yourself

1. Change the instruction to `kind="confirm"` and ask the model to confirm before greeting — observe how the asker renders a y/n prompt instead.
2. Drop the `withAskUserOnly` filter and ask the model to "find out my name from the git config" — it will reach for `bash` instead of asking, which shows why constraining the toolset matters.